### Load Trained Model

In [1]:
# %matplotlib notebook

import ipywidgets.widgets as widgets
from ipywidgets.widgets import Box, HBox, VBox, Layout, Label, Output
import traitlets

from jetbot.utils import model_selection
from jetbot import RoadCruiserTRT


Next, load the trained weights from the ``best_steering_model_xy_model.pth`` file that you uploaded.

Currently, the model weights are located on the CPU memory execute the code below to transfer to the GPU device.

We have now loaded our model, but there's a slight issue. The format that we trained our model doesn't exactly match the format of the camera. To do that, we need to do some preprocessing. This involves the following steps:

1. Convert from HWC layout to CHW layout
2. Normalize using same parameters as we did during training (our camera provides values in [0, 255] range and training loaded images in [0, 1] range so we need to scale by 255.0
3. Transfer the data from CPU memory to GPU memory
4. Add a batch dimension

In [4]:
from IPython.display import display
import ipywidgets

image_widget = ipywidgets.Image(width=300, height=300)
# fps_widget = ipywidgets.FloatText(description='Capture rate')

traitlets.dlink((RC, 'cap_image'), (image_widget, 'value'))
# traitlets.dlink((RC.camera, 'cap_time'), (fps_widget, 'value'))
                

In [5]:
speed_gain_slider = ipywidgets.FloatSlider(min=0, max=1, step=0.001, value=0.3, description='speed gain', readout_format='.3f')
steering_gain_slider = ipywidgets.FloatSlider(min=0, max=0.5, step=0.001, value=0.22, description='steering gain', readout_format='.3f')
steering_dgain_slider = ipywidgets.FloatSlider(min=0, max=2.0, step=0.001, value=1.15, description='steering kd', readout_format='.3f')
steering_bias_slider = ipywidgets.FloatSlider(min=-0.1, max=0.1, step=0.001, value=-0.01, description='steering bias', readout_format='.3f')

traitlets.dlink((speed_gain_slider, 'value'), (RC, 'speed_gain_rc'))
traitlets.dlink((steering_gain_slider, 'value'), (RC, 'steering_gain_rc'))
traitlets.dlink((steering_dgain_slider, 'value'), (RC, 'steering_dgain_rc'))
traitlets.dlink((steering_bias_slider, 'value'), (RC, 'steering_bias_rc'))

# VBox_image = ipywidgets.VBox([image_widget, fps_widget], layout=ipywidgets.Layout(align_self='center'))
VBox_image = ipywidgets.VBox([image_widget], layout=ipywidgets.Layout(align_self='center'))
VBox_control = ipywidgets.VBox([speed_gain_slider, steering_gain_slider, steering_dgain_slider, steering_bias_slider], layout=ipywidgets.Layout(align_self='center'))


In [6]:
x_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, description='x')
y_slider = ipywidgets.FloatSlider(min=0, max=2.0, orientation='vertical', description='y')
steering_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, description='steering')
speed_slider = ipywidgets.FloatSlider(min=0, max=1.0, orientation='vertical', description='speed')

traitlets.dlink((RC, 'x_slider'), (x_slider, 'value'))
traitlets.dlink((RC, 'y_slider'), (y_slider, 'value'))
traitlets.dlink((RC, 'steering_rc'), (steering_slider, 'value'))
traitlets.dlink((RC, 'speed_rc'), (speed_slider, 'value'))

Box_y_state = ipywidgets.HBox([y_slider, speed_slider])
Box_x_state = ipywidgets.VBox([x_slider, steering_slider])
Box_state = ipywidgets.VBox([Box_y_state, Box_x_state])


Cool! We've created our neural network execution function, but now we need to attach it to the camera for processing.

We accomplish that with the observe function.

Awesome! If your robot is plugged in it should now be generating new commands with each new camera frame. 

You can now place JetBot on  Lego or Track you have collected data on and see whether it can follow track.

If you want to stop this behavior, you can unattach this callback by executing the code below.

In [8]:
display(HBox([model_type_widget, model_path_widget], layout={'border': '2px solid black'}))

display(ipywidgets.HBox([Box_state, VBox_image, VBox_control]))

button_start = ipywidgets.Button(description='Start', tooltip='Click to start running', icon='play')
button_start.style.button_color='lightBlue'
button_start.on_click(start)

button_stop = ipywidgets.Button(description='Stop', tooltip='Click to stop running', icon="stop")
button_stop.style.button_color='Red'
button_stop.on_click(stop)

display(HBox([button_start, button_stop]))
out

Output(layout=Layout(border='2px solid black'))

### Conclusion
That's it for this live demo! Hopefully you had some fun seeing your JetBot moving smoothly on track following the road!!!

If your JetBot wasn't following road very well, try to spot where it fails. The beauty is that we can collect more data for these failure scenarios and the JetBot should get even better :)